In [79]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()


True

In [80]:
client = OpenAI(
    api_key=os.environ['api_key'],
    base_url=os.environ['base_url'],
)


In [81]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather", # 工具的名称： 小写、下划线， v开头
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {  # 入参属性名
                        "type": "string",  # 属性的类型： string、number、boolean
                        "description": "The city and state, e.g. San Francisco, CA",  #  参数的说明
                    }
                },
                "required": ["city_name"]  # 参数是否是必填的
            },
        }
    },
]

#定义函数
def get_weather(city_name):
    # 这里可以调用实际的天气API获取天气信息
    return f"{city_name}的天气是晴天，温度25摄氏度。"

In [82]:
def available_functions():
    return {
        "get_weather": get_weather
    }

In [89]:
messages = [
    {"role": "user", "content": "请帮我查询北京和上海的天气情况"}
]
response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages,
    tools=tools,
    parallel_tool_calls=True,  # 开启并行工具调用， 可能不同的模型支持的程度不同
)
response

ChatCompletion(id='03dc6b2b-07c5-4c58-a85e-fc4d8635a6ab', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_aWtmRJSbPTPJ68CiUH3y2137', function=Function(arguments='{"city_name": "北京"}', name='get_weather'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='call_01_hRLEXmoq3QnSODiflPYA1358', function=Function(arguments='{"city_name": "上海"}', name='get_weather'), type='function', index=1)], reasoning_content="We need to query weather for Beijing and Shanghai. We have tool get_weather requires city_name. We can call two independent tools in same block. Need respond in Chinese likely. Let's call."))], created=1790666697, model='deepseek-v4-pro', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='a307abda487cd1b463329ccb945ce396', usage=Comp

In [91]:
messages.append(response.choices[0].message.model_dump())
tool_calls = response.choices[0].message.tool_calls
tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_00_aWtmRJSbPTPJ68CiUH3y2137', function=Function(arguments='{"city_name": "北京"}', name='get_weather'), type='function', index=0),
 ChatCompletionMessageFunctionToolCall(id='call_01_hRLEXmoq3QnSODiflPYA1358', function=Function(arguments='{"city_name": "上海"}', name='get_weather'), type='function', index=1)]

In [84]:
import json

def execute_tool_calls(messages,response):

    messages.append(response.choices[0].message.model_dump())
    tool_calls = response.choices[0].message.tool_calls

    for tool_call in tool_calls:
        #拿到要调用的函数
        function_name = tool_call.function.name
        #函数的参数
        arguments = json.loads(tool_call.function.arguments)
        if function_name in available_functions():
            result = available_functions()[function_name](**arguments)
            messages.append({
                "role": "tool",   # 固定写法，表示这是工具调用的结果
                "tool_call_id": tool_call.id,   # 关联到具体的工具调用
                "content": str(result)
            })
    return messages

In [85]:
messages = execute_tool_calls(messages, response)
messages

[{'role': 'user', 'content': '请帮我查询北京和上海的天气情况'},
 {'content': '',
  'refusal': None,
  'role': 'assistant',
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_00_GoZG76Br46sF4n22B5kG9498',
    'function': {'arguments': '{"city_name": "北京"}', 'name': 'get_weather'},
    'type': 'function',
    'index': 0},
   {'id': 'call_01_iLj3LoZqGboMu84R0T460250',
    'function': {'arguments': '{"city_name": "上海"}', 'name': 'get_weather'},
    'type': 'function',
    'index': 1}],
  'reasoning_content': 'The user wants weather for Beijing and Shanghai. I need to call get_weather twice. Both are independent, so I can make both calls in the same block.'},
 {'role': 'tool',
  'tool_call_id': 'call_00_GoZG76Br46sF4n22B5kG9498',
  'content': '北京的天气是晴天，温度25摄氏度。'},
 {'role': 'tool',
  'tool_call_id': 'call_01_iLj3LoZqGboMu84R0T460250',
  'content': '上海的天气是晴天，温度25摄氏度。'}]

In [87]:
second_response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages
)

second_response.choices[0].message.content

'北京和上海目前的天气情况如下：\n\n| 城市 | 天气 | 温度 |\n|------|------|------|\n| 北京 | 晴天 | 25℃ |\n| 上海 | 晴天 | 25℃ |\n\n两个城市的天气都很好，气温适宜。'

In [88]:
from IPython.display import Markdown
display(Markdown(second_response.choices[0].message.content))

北京和上海目前的天气情况如下：

| 城市 | 天气 | 温度 |
|------|------|------|
| 北京 | 晴天 | 25℃ |
| 上海 | 晴天 | 25℃ |

两个城市的天气都很好，气温适宜。